# MoCo on CIFAR-10

这个 Notebook 展示 `MoCo` 在 `CIFAR-10` 上的一个教学版实现，重点解释：

- query encoder 和 key encoder 的分工
- momentum update 为什么存在
- queue / memory bank 如何提供大量负样本
- InfoNCE loss 如何计算
- `MoCo` 和 `SimCLR` 的关键差异

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# dataclass 用于集中管理实验配置
from dataclasses import dataclass

# matplotlib 用于样本和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
# torchvision 提供数据集、增强和 backbone
from torchvision import datasets, models, transforms

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    data_root: str = './data'
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 2
    lr: float = 1e-3
    epochs: int = 5
    feature_dim: int = 128
    queue_size: int = 4096
    momentum: float = 0.999
    temperature: float = 0.07
    linear_batch_size: int = 128
    linear_epochs: int = 3


cfg = Config()
cfg

## 2. 数据增强与双视图构造

MoCo 也是对比学习方法，因此仍然需要把同一张图片增强成两个视图。

区别在于：
- SimCLR 主要依赖“大 batch 中其他样本”作为负样本
- MoCo 通过一个动态队列保存历史特征，避免强依赖超大 batch

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# 对比学习常用较强增强
contrastive_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomResizedCrop(cfg.image_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 线性评估阶段使用较普通的预处理
eval_train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

eval_test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

In [ ]:
class PairDataset(Dataset):
    def __init__(self, root, train=True, transform=None, download=True):
        self.dataset = datasets.CIFAR10(root=root, train=train, download=download)
        self.transform = transform
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        view1 = self.transform(image)
        view2 = self.transform(image)
        return view1, view2, label


contrastive_dataset = PairDataset(cfg.data_root, train=True, transform=contrastive_transform, download=True)
linear_train_dataset = datasets.CIFAR10(cfg.data_root, train=True, transform=eval_train_transform, download=True)
linear_test_dataset = datasets.CIFAR10(cfg.data_root, train=False, transform=eval_test_transform, download=True)
classes = contrastive_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


view1, view2, label = contrastive_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(denormalize(view1, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1))
axes[0].set_title(f'query view / {classes[label]}')
axes[0].axis('off')
axes[1].imshow(denormalize(view2, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1))
axes[1].set_title(f'key view / {classes[label]}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
contrastive_loader = DataLoader(
    contrastive_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

linear_train_loader = DataLoader(
    linear_train_dataset,
    batch_size=cfg.linear_batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

linear_test_loader = DataLoader(
    linear_test_dataset,
    batch_size=cfg.linear_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

q_images, k_images, labels = next(iter(contrastive_loader))
print('query batch shape:', q_images.shape)
print('key batch shape:', k_images.shape)
print('label batch shape:', labels.shape)

## 3. MoCo 模型结构

MoCo 由三部分组成：

1. `query encoder`
   - 当前 batch 的查询分支。

2. `key encoder`
   - 用动量方式从 query encoder 同步参数，生成更稳定的 key 特征。

3. `queue`
   - 保存历史 key 特征，提供大量负样本。

In [ ]:
class EncoderWithProjector(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=None)
        in_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projector = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, out_dim),
        )
        self.feature_dim = in_dim

    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return h, z


class MoCo(nn.Module):
    def __init__(self, feature_dim=128, queue_size=4096, momentum=0.999, temperature=0.07):
        super().__init__()
        self.query_encoder = EncoderWithProjector(out_dim=feature_dim)
        self.key_encoder = EncoderWithProjector(out_dim=feature_dim)
        self.momentum = momentum
        self.temperature = temperature
        self.feature_dim = feature_dim
        self.backbone_feature_dim = self.query_encoder.feature_dim

        # 初始化时让 key encoder 和 query encoder 参数完全一致
        for param_q, param_k in zip(self.query_encoder.parameters(), self.key_encoder.parameters()):
            param_k.data.copy_(param_q.data)
            param_k.requires_grad = False

        # queue 里保存历史 key 特征，形状是 [D, K]
        self.register_buffer('queue', F.normalize(torch.randn(feature_dim, queue_size), dim=0))
        self.register_buffer('queue_ptr', torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def momentum_update_key_encoder(self):
        # key encoder 不是直接反向传播更新，而是用 query encoder 做动量更新
        for param_q, param_k in zip(self.query_encoder.parameters(), self.key_encoder.parameters()):
            param_k.data = param_k.data * self.momentum + param_q.data * (1.0 - self.momentum)

    @torch.no_grad()
    def dequeue_and_enqueue(self, keys):
        batch_size = keys.shape[0]
        ptr = int(self.queue_ptr)
        queue_size = self.queue.shape[1]

        # 这里假设 queue_size 能被 batch_size 整除，便于演示实现
        self.queue[:, ptr:ptr + batch_size] = keys.T
        ptr = (ptr + batch_size) % queue_size
        self.queue_ptr[0] = ptr

    def forward(self, im_q, im_k):
        # query 分支正常参与梯度更新
        q_h, q = self.query_encoder(im_q)

        with torch.no_grad():
            # 先更新 key encoder，再提取 key 特征
            self.momentum_update_key_encoder()
            k_h, k = self.key_encoder(im_k)

        # 正样本相似度：当前 q 对当前 k
        l_pos = torch.einsum('nc,nc->n', [q, k]).unsqueeze(-1)
        # 负样本相似度：当前 q 对 queue 里的历史 key
        l_neg = torch.einsum('nc,ck->nk', [q, self.queue.clone().detach()])

        # 把正样本放在第 0 列，其余列是负样本
        logits = torch.cat([l_pos, l_neg], dim=1)
        logits = logits / self.temperature

        # 分类目标永远是第 0 列，也就是正样本位置
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)

        # 当前 batch 的 key 会在本轮 loss 计算后入队，供后续 batch 使用
        self.dequeue_and_enqueue(k)
        return q_h, q, k_h, k, logits, labels


model = MoCo(
    feature_dim=cfg.feature_dim,
    queue_size=cfg.queue_size,
    momentum=cfg.momentum,
    temperature=cfg.temperature,
).to(device)
model

## 4. MoCo 的 loss 怎么看

MoCo 使用的是一类 `InfoNCE` 风格损失。

对每个 query `q` 来说：
- 正样本是当前 batch 对应的 `k`
- 负样本是 queue 中保存的大量历史 key

单个样本的损失可以写成：

$$
\ell = -\log \frac{\exp(q \cdot k^+ / \tau)}{\exp(q \cdot k^+ / \tau) + \sum_{i=1}^{K} \exp(q \cdot k_i^- / \tau)}
$$

其中：
- `k^+` 是正样本 key
- `k_i^-` 是 queue 中的负样本 key
- `tau` 是温度系数

和 SimCLR 相比，MoCo 的关键区别不是损失公式本身，而是负样本的来源：
- SimCLR 主要来自同一大 batch
- MoCo 主要来自历史队列

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.query_encoder.parameters(), lr=cfg.lr)


q_images, k_images, _ = next(iter(contrastive_loader))
q_images = q_images.to(device)
k_images = k_images.to(device)

with torch.no_grad():
    _, q, _, k, logits, labels = model(q_images, k_images)

print('q shape:', q.shape)
print('k shape:', k.shape)
print('queue shape:', model.queue.shape)
print('logits shape:', logits.shape)
print('labels shape:', labels.shape)
print('sample loss:', float(criterion(logits, labels)))

## 5. 训练函数

In [ ]:
def train_one_epoch_moco(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0

    for im_q, im_k, _ in dataloader:
        im_q = im_q.to(device)
        im_k = im_k.to(device)

        optimizer.zero_grad()
        _, _, _, _, logits, labels = model(im_q, im_k)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = im_q.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total

In [ ]:
history = []

for epoch in range(cfg.epochs):
    epoch_loss = train_one_epoch_moco(model, contrastive_loader, criterion, optimizer, device)
    history.append(epoch_loss)
    print(f'Epoch [{epoch + 1}/{cfg.epochs}] moco_loss={epoch_loss:.4f}')

In [ ]:
epochs = range(1, len(history) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history, marker='o')
plt.title('MoCo contrastive loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 6. 提取 encoder 特征并做线性评估

和 SimCLR 一样，真正更关心的通常是 encoder 学到的表示质量。

In [ ]:
@torch.no_grad()
def extract_features(backbone, dataloader, device):
    backbone.eval()
    features = []
    labels = []
    for images, target in dataloader:
        images = images.to(device)
        feats = backbone(images)
        features.append(feats.cpu())
        labels.append(target)
    return torch.cat(features, dim=0), torch.cat(labels, dim=0)


train_features, train_labels = extract_features(model.query_encoder.backbone, linear_train_loader, device)
test_features, test_labels = extract_features(model.query_encoder.backbone, linear_test_loader, device)

print('train_features:', train_features.shape)
print('test_features:', test_features.shape)

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features.float()
        self.labels = labels.long()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


train_feature_loader = DataLoader(FeatureDataset(train_features, train_labels), batch_size=cfg.linear_batch_size, shuffle=True)
test_feature_loader = DataLoader(FeatureDataset(test_features, test_labels), batch_size=cfg.linear_batch_size, shuffle=False)

linear_head = nn.Linear(model.backbone_feature_dim, 10).to(device)
linear_optimizer = optim.Adam(linear_head.parameters(), lr=1e-3)
linear_criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch_linear(head, dataloader, criterion, optimizer, device):
    head.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = head(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * features.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate_linear(head, dataloader, criterion, device):
    head.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)
        logits = head(features)
        loss = criterion(logits, labels)
        running_loss += loss.item() * features.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, running_correct / total

In [ ]:
for epoch in range(cfg.linear_epochs):
    train_loss, train_acc = train_one_epoch_linear(linear_head, train_feature_loader, linear_criterion, linear_optimizer, device)
    test_loss, test_acc = evaluate_linear(linear_head, test_feature_loader, linear_criterion, device)
    print(
        f'Epoch [{epoch + 1}/{cfg.linear_epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'test_loss={test_loss:.4f} test_acc={test_acc:.4f}'
    )

## 7. MoCo 和 SimCLR 的差异

- SimCLR 更依赖大 batch，因为负样本主要来自同 batch。
- MoCo 用 queue 保存历史特征，所以能在较小 batch 下拥有大量负样本。
- MoCo 需要 key encoder 和动量更新机制，以保证队列里特征分布相对稳定。
- SimCLR 结构更直接；MoCo 在工程上更强调负样本缓存机制。